# Exercise 2.2.5 — Merging & Combining Datasets

This exercise reads the feature table from **Exercise 2.2.4** (`data/20_processed/datania_households_features.csv`) and enriches it by joining reference tables, then stacking survey waves.

You will practice:
- Combining tables with `pd.merge()` and choosing the right **join type**
- **Join-key hygiene**: matching dtypes, trimming whitespace, counting missing keys
- Auditing a merge with `indicator=True` and enforcing expectations with `validate=`
- Recognising **cardinality** problems (row explosions, many-to-many merges)
- **Post-merge validation** and saving the final table
- Stacking waves/modules with `pd.concat()`

> **Pipeline:** run Exercises 2.2.3 and 2.2.4 first. This notebook writes the final merged table to `20_processed/`.

### Path Setup (run first)

In [ ]:
import os
import numpy as np
import pandas as pd

features_path = '../../data/20_processed/datania_households_features.csv'
df = pd.read_csv(features_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Households:', df.shape)
df[['hh_id', 'region_code', 'province_name', 'district', 'education_code']].head()

---

## Task 1 — A basic left merge with a region lookup

Geographic labels usually live in a separate reference table. Build a small `region_lookup` and attach `region_name` to every household with a **left join** (keep all households).

In [ ]:
region_lookup = pd.DataFrame({
    'region_code': ['01', '02', '03', '04', '05', '06'],
    'region_name': ['Eastern', 'Northern', 'Central', 'Southern', 'Western', 'Highland'],
})

merged = pd.merge(df, region_lookup, on='region_code', how= # your code here — 'left' )
print('Before:', df.shape, '-> After:', merged.shape)
merged[['hh_id', 'region_code', 'region_name']].head(10)

**Questions:**

- Did the row count change? For a left join with a unique key on the right, what should happen to it?
- Some households get a `NaN` `region_name`. Which households, and why (think about what 2.2.3 did to `region_code`)?

---

## Task 2 — Join-key hygiene

Most "failed merge" mysteries are a key that is text on one side and a number on the other, or a stray space. Before joining, make both keys the **same dtype**, trim whitespace, and count missing keys on each side.

In [ ]:
df['region_code'] = df['region_code'].astype('string').str.strip()
region_lookup['region_code'] = region_lookup['region_code'].astype('string').str.strip()

print('Missing keys (households):', df['region_code']. # your code here — isna().sum() )
print('Missing keys (lookup):    ', region_lookup['region_code'].isna().sum())

**Questions:**

- How many household rows have a missing join key? After a left join, what value will their `region_name` take?
- Why is fixing the key *at the source* better than dropping the unmatched rows afterwards?

---

## Task 3 — Choose the join type, then audit with `indicator`

The join type decides which rows survive. Add `indicator=True` to label every row as `left_only`, `right_only`, or `both`.

In [ ]:
merged = pd.merge(df, region_lookup, on='region_code', how='left', indicator= # your code here )
print(merged['_merge'].value_counts())

In [ ]:
inner = pd.merge(df, region_lookup, on='region_code', how='inner')
outer = pd.merge(df, region_lookup, on='region_code', how='outer')
print('inner:', inner.shape[0], '| left:', merged.shape[0], '| outer:', outer.shape[0])

**Questions:**

- What does each `left_only` row represent here? Why is that group worth reporting back to the data team?
- The inner join has fewer rows than the left join. Which households did it drop?

---

## Task 4 — Cardinality: row explosions and `validate`

If the right-hand key is **not unique**, every duplicate match multiplies rows. `validate=` makes your assumption explicit and raises an error if it is violated.

In [ ]:
# A lookup that accidentally lists region '01' twice
bad_lookup = pd.DataFrame({
    'region_code': ['01', '01', '02'],
    'region_name': ['Eastern', 'Eastern (dup)', 'Northern'],
})

exploded = pd.merge(df, bad_lookup, on='region_code', how='left')
print('Rows before:', len(df), '-> after bad merge:', len(exploded))

In [ ]:
# Make the expectation explicit: each household should match at most one region.
try:
    pd.merge(df, bad_lookup, on='region_code', how='left', validate= # your code here — 'many_to_one' )
except Exception as e:
    print(type(e).__name__, '->', e)

**Questions:**

- Why did the row count grow? Which households were duplicated?
- What do `'one_to_one'`, `'one_to_many'`, and `'many_to_one'` each promise? Which fits a household -> region lookup?

---

## Task 5 — Merge on differently-named keys

When the key column has a different name on each side, use `left_on` / `right_on`. Attach a full education label from a lookup keyed on `code`.

In [ ]:
education_lookup = pd.DataFrame({
    'code': [1, 2, 3, 4],
    'education_label_full': ['No schooling', 'Primary', 'Secondary', 'Tertiary'],
})

merged_edu = pd.merge(
    df,
    education_lookup,
    left_on= # your code here — 'education_code'
    right_on='code',
    how='left',
)
merged_edu[['hh_id', 'education_code', 'education_label_full']].head()

**Question:** `education_code` is a float (it carries `NaN`) while `code` is an integer. The merge still matches — why? When would a dtype mismatch silently produce *no* matches instead?

---

## Task 6 — Many-to-many: a warning

When **both** sides have duplicate keys, pandas produces the full cross-product — often nonsense. The fix is a safer (compound) key, not "merge harder".

In [ ]:
roster = pd.DataFrame({'hh_id': [1, 1], 'person': ['A', 'B']})
visits = pd.DataFrame({'hh_id': [1, 1], 'visit': ['V1', 'V2']})

mm = pd.merge(roster, visits, on='hh_id')
print(len(mm), 'rows from two 2-row tables')
mm

**Questions:**

- Why did two people and two visits become four rows?
- What compound key would make each row identify exactly one record?

---

## Task 7 — Post-merge validation, then save the final table

A merge is finished only when you have confirmed the result. Check the row count, key uniqueness, and the unmatched rate, then save to `20_processed/`.

In [ ]:
final = pd.merge(df, region_lookup, on='region_code', how='left')

print('Before:', len(df), '| After:', len(final))
print('Duplicate hh_id:', final['hh_id'].duplicated().sum())
print('Unmatched region:', round(final['region_name'].isna().mean(), 3))

In [ ]:
final = final.reset_index(drop=True)
out_path = '../../data/20_processed/datania_households_merged.csv'

final.to_csv( # your code here — index=False )
print('Saved:', out_path, '|', final.shape)

---

## Task 8 — Appending waves with `concat()`

Merging combines **columns**; appending combines **rows**. When you collect the same survey across waves, stack them with `pd.concat()` and add a column recording where each row came from.

In [ ]:
wave1 = df[['hh_id', 'region_code', 'income_dkw']].head(3).assign(wave='2024')
wave2 = pd.DataFrame({
    'hh_id': ['HH0101', 'HH0102'],
    'region_code': ['01', '02'],
    'income_dkw': [62000.0, 47000.0],
}).assign(wave='2025')

combined = pd.concat([wave1, wave2], ignore_index= # your code here — True )
combined

**Questions:**

- Why add a `wave` column before stacking?
- `concat()` aligns on column names and fills gaps with `NaN` without warning. What would happen if `wave2` had `income` instead of `income_dkw`? How do you guard against that schema drift?